# LDM experiments: train, review, and release

**How to use this notebook:** Run setup sections 1–4, then **Start** (5) trains every entry in `RUN_CONFIGS` sequentially. Use **Inspect/Resume** (6–7) for a particular saved run. Metrics (8), snapshot validation (9), generation review (10), and optional analyses (11) help assess a run. Publishing (12–13) is a separate choice. New run checkpoints and reports live on Drive; temporary data caches and snapshots live in `/content/ldm`. The older `runs/with_subd` folder is not changed.


## 1. Load the GitHub code

**Clone/update cell:** Reads `GITHUB_TOKEN` from Colab Secrets, clones or pulls `appodoom/ldm` into `/content/ldm`, and prints the exact code commit. It does not start training. The next cell installs the package and runs its contract tests.


In [ ]:
import base64
import os
import subprocess
import sys
from pathlib import Path
from google.colab import userdata

project = Path("/content/ldm")
token = userdata.get("GITHUB_TOKEN")
assert token, "Add GITHUB_TOKEN to Colab Secrets and allow notebook access"
auth = base64.b64encode(f"x-access-token:{token}".encode()).decode()
git_env = os.environ.copy()
git_env.update({
    "GIT_CONFIG_COUNT": "1",
    "GIT_CONFIG_KEY_0": "http.https://github.com/.extraheader",
    "GIT_CONFIG_VALUE_0": f"AUTHORIZATION: basic {auth}",
})
if (project / ".git").exists():
    subprocess.run(["git", "-C", str(project), "pull", "--ff-only", "origin", "main"],
                   env=git_env, check=True)
elif project.exists():
    raise RuntimeError("/content/ldm exists but is not the Git checkout")
else:
    subprocess.run(["git", "clone", "https://github.com/appodoom/ldm.git", str(project)],
                   env=git_env, check=True)
code_commit = subprocess.check_output(
    ["git", "-C", str(project), "rev-parse", "HEAD"], text=True).strip()
print("Code commit:", code_commit)


### Install and check the package

The next cell installs the cloned code into this Colab runtime and runs the repository tests. It stops if those tests fail.


In [ ]:
os.chdir(project)
print("Python:", sys.version.split()[0])
subprocess.run([sys.executable, "-m", "pip", "install", "-e", "."], check=True)
subprocess.run([sys.executable, "-m", "unittest", "discover", "-s", "tests", "-v"],
               check=True)


## 2. Load data and persistent storage

**HF dataset cell:** Signs in to Hugging Face and downloads the pinned five-line normal and decorated corpus into `/content/ldm/.derbake`. These source files are read during preparation; training does not edit them. **Drive cell:** Mounts `MyDrive/LDM` so run checkpoints, logs, reviews, and release records survive a Colab reset.


In [ ]:
from huggingface_hub import notebook_login, snapshot_download

notebook_login()
data_root = Path(snapshot_download(
    repo_id="Appodoom/ldm-data", repo_type="dataset",
    revision="34fb2449a25472e856a4db0aa85e280e9a82a8b2",
    allow_patterns="five-line/**", local_dir=project / ".derbake",
))
ordinary = data_root / "five-line" / "normal"
decorated = data_root / "five-line" / "decorated"
assert len(list(ordinary.glob("*.derbake"))) == 116
assert len(list(decorated.glob("*.derbake"))) == 116
print("Paired files:", len(list(ordinary.glob("*.derbake"))))


### Mount Google Drive

The next cell sets the durable root for new runs, reviews, plots, and release records. `/content` data can disappear when the runtime ends.


In [ ]:
from google.colab import drive
import torch, json
drive.mount('/content/drive')
storage_root = Path('/content/drive/MyDrive/LDM')
runs_root = storage_root / 'runs'
runs_root.mkdir(parents=True, exist_ok=True)
cache_root = project / 'datasets'
print('Runs:', runs_root, '| Device:', 'cuda' if torch.cuda.is_available() else 'cpu')


## 3. Define the run list

Keep every experiment in the single `RUN_CONFIGS` list below. Add another `RunConfig(...)` entry whenever you want another combination of data, model, and training settings. `DataConfig.variant` inside each entry chooses `with_subd` or `no_subd`; there is no separate variant list. The preparation and training cells process every entry in list order. This cell validates settings and checks optional custom `name` values for duplicates. Without a name, the run gets the old automatic timestamp/settings/hash name. Nothing trains here.


In [ ]:
from ldm.experiment_config import (
    Activation, AttentionLayout, DatasetVariant, PositionEncoding,
    DataConfig, ModelConfig, TrainConfig, EvalConfig, RunConfig, StructuredStreams,
)
from ldm.experiment_workflow import check_run_name

RUN_CONFIGS = [
    # Optional: add name='my_run' inside a RunConfig to use a custom name.
    RunConfig(data=DataConfig(variant=DatasetVariant.WITH_SUBD, context_size=1024, stride=512, max_beats_per_cycle=16, include_decorated=True, decorated_weight=1.0), model=ModelConfig(layers=4, heads=4, embeddings=256, mlp_ratio=4, dropout=0.1, attention=AttentionLayout.ALTERNATING_GLOBAL_STRUCTURED, activation=Activation.TANH, position=PositionEncoding.ROPE, rope_theta=10000.0, rope_fraction=1.0, use_type_embeddings=True, concat_type_embeddings=False, streams=StructuredStreams(inside_beat=True, beat_memory=True, cycle_memory=True)), train=TrainConfig(epochs=10, batch_size=4, learning_rate=0.0003, weight_decay=0.01, grad_clip=1.0, amp=True, save_every_steps=200, log_every_steps=50, early_stopping=False), evaluation=EvalConfig(batch_size=4, every_steps=200, save_plots=True)),
    RunConfig(data=DataConfig(variant=DatasetVariant.NO_SUBD, context_size=1024, stride=512)),
    # Add more RunConfig(...) entries here; each chooses its variant in DataConfig.
]
assert RUN_CONFIGS, 'Add at least one RunConfig'
for index, run_config in enumerate(RUN_CONFIGS):
    print(f'Config {index}: {check_run_name(run_config, runs_root, RUN_CONFIGS)}')
    print(json.dumps(run_config.to_dict(), indent=2))
    print('Config hash:', run_config.fingerprint)


## 4. Prepare or reuse the dataset

This cell loops over every `RunConfig` and reads its data settings and the pinned source corpus. It splits complete cycles **within each file** about 70/20/10, keeps normal/decorated pairs aligned, and creates model arrays under `/content/ldm/datasets/<variant>-<recipe-hash>`. It reuses that cache only when the full recipe and source bytes match. The cache is temporary and can be rebuilt.


In [ ]:
from ldm.experiment_workflow import prepare_or_load_dataset
PREPARED_CONFIGS = tuple(RUN_CONFIGS)
PREPARED_DATASETS = []
for index, run_config in enumerate(RUN_CONFIGS):
    print(f'Preparing config {index + 1}/{len(RUN_CONFIGS)}: {run_config.data.variant.value}')
    prepared = prepare_or_load_dataset(run_config, ordinary, decorated, cache_root)
    PREPARED_DATASETS.append(prepared)
    manifest = json.loads((prepared / 'dataset_manifest.json').read_text(encoding='utf-8'))
    print('Dataset:', prepared)
    print(json.dumps(manifest['stats'], indent=2))


## 5. Start new training runs

Run this cell only when you want **new weights for every configuration**. It creates a separate Drive folder under `MyDrive/LDM/runs` for each entry and trains them sequentially with live progress. `last.pt` is for resuming; `best.pt` is chosen by validation loss. The folder also receives the saved config, logs, evaluation history, and final test metrics. Rerunning this cell starts another set of automatically named runs. A custom `name` can be used only once; to continue that run, use Resume in section 7.


In [ ]:
from ldm.experiment_workflow import start_run
assert (
    'PREPARED_CONFIGS' in globals()
    and PREPARED_CONFIGS == tuple(RUN_CONFIGS)
    and len(PREPARED_DATASETS) == len(RUN_CONFIGS)
), 'Run preparation again after changing RUN_CONFIGS'
STARTED_RUNS = []
for index, (run_config, prepared) in enumerate(zip(RUN_CONFIGS, PREPARED_DATASETS)):
    print(f'Starting config {index + 1}/{len(RUN_CONFIGS)}: {run_config.data.variant.value}')
    run = start_run(run_config, prepared, runs_root)
    STARTED_RUNS.append(run)
    print('Completed run:', run.name)
print('Runs started:', [path.name for path in STARTED_RUNS])


## 6. List and inspect Drive runs

This read-only cell lists named runs, status, steps, and best loss. Set `SELECT_RUN_ID` to inspect a particular run, including one interrupted before the Start cell returned. By default it selects the last run completed by section 5. The later review and release cells use this selection; training itself processes the entire list. It does not load model weights or change the run. The older fixed-path `runs/with_subd` folder is left untouched.


In [ ]:
from ldm.experiment_workflow import list_runs, inspect_run
for item in list_runs(runs_root):
    print(item['run_id'], item['status'], item.get('last_step'), item.get('best_val_loss'))
SELECT_RUN_ID = STARTED_RUNS[-1].name if globals().get('STARTED_RUNS') else None  # Or type any existing run ID.
if SELECT_RUN_ID:
    run = runs_root / SELECT_RUN_ID
    print(json.dumps(inspect_run(run), indent=2)[:12000])


## 7. Resume a selected run

This cell is **off by default** (`RESUME_NOW = False`). Set `SELECT_RUN_ID` and enable it to continue from that run's `last.pt`. It reads the saved config from Drive, not the currently selected notebook preset. `TARGET_EPOCHS` means total epochs, including completed ones; a lower target is rejected.


In [ ]:
from ldm.experiment_workflow import resume_run
RESUME_NOW = False
TARGET_EPOCHS = 10
if RESUME_NOW:
    assert SELECT_RUN_ID, 'Select an existing run ID first'
    result = resume_run(runs_root / SELECT_RUN_ID, target_epochs=TARGET_EPOCHS)
    print('Stopped:', result['stop_reason'], 'at step', result['steps'])
else:
    print('Resume skipped. Set RESUME_NOW=True when ready.')


## 8. Inspect metrics and plots

This cell reads the selected run's final `metrics.json`, prints key test measures, saves a training/validation loss plot in the run folder, and writes a comparison CSV under `MyDrive/LDM/comparisons`. Validation chose `best.pt`; test metrics describe that checkpoint. Test cycles come from files also represented in training.


In [ ]:
from ldm.experiment_workflow import compare_runs
assert SELECT_RUN_ID, 'Select a run first'
run = runs_root / SELECT_RUN_ID
metrics = json.loads((run / 'metrics.json').read_text(encoding='utf-8'))
print(json.dumps({k: metrics[k] for k in ('steps', 'best_step', 'best_val_loss', 'test_loss', 'stop_reason')}, indent=2))
print('Test token accuracy:', metrics['test_report']['combined']['token_accuracy'])
print('Test HIT accuracy:', metrics['test_report']['combined']['hit_accuracy'])
print('Test skeleton HIT accuracy:', metrics['test_report']['combined']['skeleton_hit_accuracy'])
print(json.dumps(compare_runs([runs_root / item['run_id'] for item in list_runs(runs_root) if (runs_root / item['run_id'] / 'metrics.json').exists()]), indent=2))
from ldm.report_ldm import save_history_plot, save_comparison
if inspect_run(run)['config']['evaluation']['save_plots']:
    print('Plot:', save_history_plot(run))
selected_runs = [runs_root / item['run_id'] for item in list_runs(runs_root) if (runs_root / item['run_id'] / 'metrics.json').exists()]
if selected_runs:
    print('Comparison CSV:', save_comparison(selected_runs, storage_root / 'comparisons' / 'runs.csv'))


## 9. Build a local Hugging Face snapshot

This cell exports the selected run's `best.pt` into `/content/ldm/snapshots/...` with weights, model code, tokenizer, processor, and metadata. An isolated check loads `AutoModel` and `AutoProcessor` and verifies a five-line round trip. The snapshot is temporary; **this cell does not upload it**. Run it before generation review or release.


In [ ]:
import hashlib
variant = json.loads((run / 'run_config.json').read_text())['model_config']['dataset_variant']
best_checkpoint = run / 'best.pt'
checkpoint_sha256 = hashlib.sha256(best_checkpoint.read_bytes()).hexdigest()
snapshot = project / 'snapshots' / f'{variant}-{checkpoint_sha256[:12]}-{code_commit[:8]}'
release_manifest_path = snapshot / 'metadata' / 'release_manifest.json'
if snapshot.exists() and not release_manifest_path.exists():
    raise RuntimeError(f'Incomplete snapshot: {snapshot}')
if not snapshot.exists():
    subprocess.run([sys.executable, '-m', 'ldm.cli', 'snapshot', '--run', str(run), '--output', str(snapshot)], check=True)
release_manifest = json.loads(release_manifest_path.read_text(encoding='utf-8'))
assert release_manifest['source_checkpoint_sha256'] == checkpoint_sha256
subprocess.run([sys.executable, '-m', 'ldm.cli', 'validate', '--snapshot', str(snapshot)], check=True)
print('Validated:', snapshot)


## 10. Optional generation review

This is an **inference check**, not training or the automatic test-set evaluation. Set `PROMPT_SOURCE_FILE` and `PROMPT_CYCLE_INDEX` below; the example uses cycle 0 of the first alphabetically sorted normal file. Change `REVIEW_SEEDS` and `InferenceConfig` to try other outputs. The example writes that cycle as a canonical one-cycle prompt; you may also put your own complete files in `REVIEW_PROMPTS`. The selected **run snapshot**, regardless of the currently selected preset, supplies the right processor: `with_subd` uses explicit `SUBD_n` model tokens, while `no_subd` infers beat subdivision from its HIT tokens. Both variants read and write the same five-line `.derbake` format, including the subdivision markers on line 5. The default first cycle is only a demonstration prompt, not a held-out quality benchmark. Each successful attempt saves the prompt, **generated-only** `.derbake`, settings, and accepted/discarded-cycle report under `MyDrive/LDM/reviews/<run-id>`. You can skip this cell.


In [ ]:
from ldm.review_ldm import InferenceConfig, review_generation
from ldm.derbake_ldm import RhythmSequence, parse_derbake, write_derbake
PROMPT_SOURCE_FILE = sorted(ordinary.glob('*.derbake'))[0]  # Choose another five-line file if desired.
PROMPT_CYCLE_INDEX = 0  # Zero-based cycle within that file.
source_rhythm = parse_derbake(PROMPT_SOURCE_FILE).rhythm
assert 0 <= PROMPT_CYCLE_INDEX < len(source_rhythm.cycles)
prompt_dir = storage_root / 'prompts' / SELECT_RUN_ID
prompt_dir.mkdir(parents=True, exist_ok=True)
prompt_file = prompt_dir / f'{PROMPT_SOURCE_FILE.stem}_cycle{PROMPT_CYCLE_INDEX + 1}.derbake'
write_derbake(RhythmSequence((source_rhythm.cycles[PROMPT_CYCLE_INDEX],)), prompt_file)
REVIEW_PROMPTS = [prompt_file]  # Add more valid prompts here.

print('Selected model variant:', variant, '| Prompt:', prompt_file)
REVIEW_SEEDS = [1337, 2026]
for prompt in REVIEW_PROMPTS:
    for seed in REVIEW_SEEDS:
        report = review_generation(snapshot, prompt, storage_root / 'reviews' / SELECT_RUN_ID,
                                   InferenceConfig(num_cycles=1, seed=seed, max_new_tokens=2048))
        print(report['output'], report['generation'])


## 11. Optional audio and embedding analysis

Provide four WAV sample directories in `SAMPLE_DIRS` to render the last generated file; leaving it empty skips audio. Decorated hits use their base sound. The cell also prints token embedding neighbors. Set `RUN_CONTEXTUAL_ANALYSIS = True` only if you want a bounded PCA table saved in the run's `analysis` folder. These analyses do not change weights.


In [ ]:
from ldm.review_ldm import render_wav, embedding_neighbors
SAMPLE_DIRS = {}  # e.g. {'doums': Path(...), 'taks': Path(...), 'tiks': Path(...), 'pa2s': Path(...)}
if SAMPLE_DIRS:
    generated_file = Path(report['output'])
    print(render_wav(generated_file, SAMPLE_DIRS, generated_file.with_suffix('.wav')))
else:
    print('Audio skipped: provide four sample directories to enable it.')
print(json.dumps(embedding_neighbors(snapshot, ['HIT_D', 'HIT_D_decorated'] +
    (['SUBD_1', 'SUBD_2'] if variant == 'with_subd' else [])), indent=2))

from ldm.review_ldm import contextual_projection
RUN_CONTEXTUAL_ANALYSIS = False  # Bounded optional CPU analysis.
if RUN_CONTEXTUAL_ANALYSIS:
    selected_dataset = Path(inspect_run(run)['dataset_path'])
    assert selected_dataset.exists(), 'Reprepare the selected run dataset in this runtime first'
    print(contextual_projection(snapshot, selected_dataset, run / 'analysis'))


## 12. Optional Hugging Face release

This cell shows files that a release would replace or remove in the matching model repo. `PUBLISH_TO_HF = False` means **nothing is uploaded**. After reviewing the snapshot and outputs, set it to `True` to upload the selected `with_subd` or `no_subd` snapshot. Training checkpoints stay on Drive; the HF model repo receives the loadable release package.


In [ ]:
from huggingface_hub import HfApi
from transformers import AutoModel, AutoProcessor
repo_id = {'with_subd': 'Appodoom/ldm-with-subd', 'no_subd': 'Appodoom/ldm-no-subd'}[variant]
api = HfApi()
remote_info = api.repo_info(repo_id=repo_id, repo_type='model')
remote_files = {item.rfilename for item in remote_info.siblings}
local_files = {path.relative_to(snapshot).as_posix() for path in snapshot.rglob('*') if path.is_file()}
stale_files = sorted(remote_files - local_files - {'.gitattributes'})
print('Remote files to remove on release:', stale_files)
PUBLISH_TO_HF = False
published_commit = None
if PUBLISH_TO_HF:
    assert metrics['stop_reason'] in {'completed', 'early_stopped'}
    assert release_manifest['source_checkpoint_sha256'] == hashlib.sha256(best_checkpoint.read_bytes()).hexdigest()
    subprocess.run([sys.executable, '-m', 'ldm.cli', 'validate', '--snapshot', str(snapshot)], check=True)
    uploaded = api.upload_folder(repo_id=repo_id, repo_type='model', folder_path=str(snapshot),
                                 commit_message=f'Release {variant} model',
                                 parent_commit=remote_info.sha, delete_patterns=stale_files)
    published_commit = uploaded.oid
    print(f'https://huggingface.co/{repo_id}/tree/{published_commit}')
else:
    print('Upload skipped.')


## 13. Verify the uploaded commit

If section 12 uploaded a model, this cell downloads that exact HF commit, runs isolated loading/round-trip checks, and saves the repo ID and commit revision under `MyDrive/LDM/releases`. Consumers should load `AutoModel` and `AutoProcessor` from the same verified revision. If no upload occurred, this cell skips.


In [ ]:
if published_commit is None:
    print('Remote validation skipped.')
else:
    remote_snapshot = Path(snapshot_download(repo_id=repo_id, repo_type='model',
        revision=published_commit, local_dir=project / 'remote_validation' / variant / published_commit))
    subprocess.run([sys.executable, '-m', 'ldm.cli', 'validate', '--snapshot', str(remote_snapshot)], check=True)
    remote_model = AutoModel.from_pretrained(repo_id, revision=published_commit, trust_remote_code=True).eval()
    remote_processor = AutoProcessor.from_pretrained(repo_id, revision=published_commit, trust_remote_code=True)
    assert remote_model.config.vocab_hash == remote_processor.tokenizer.atomic.hash
    release_record = {'repo_id': repo_id, 'revision': published_commit, 'run_id': SELECT_RUN_ID,
        'code_commit': code_commit, 'checkpoint_sha256': checkpoint_sha256,
        'dataset_fingerprint': metrics['dataset_fingerprint'],
        'metric_version': metrics['test_report']['metric_version']}
    record_path = storage_root / 'releases' / f'{variant}-{published_commit[:12]}.json'
    record_path.parent.mkdir(parents=True, exist_ok=True)
    record_path.write_text(json.dumps(release_record, indent=2) + '\n', encoding='utf-8')
    print('Verified:', published_commit, '| Record:', record_path)
